# CEE/ENV 586 
## Fall 2023

# Point Observations Data Access

This notebook provides some example functionality of the hydrodata Python package for querying the point observations data that we regularly collect and store at Princeton.

This Python package is available on Verde from within the parflow-ml module.

In [ ]:
import hydrodata.point_observations.pandas.collect_observations as point_obs
import hydrodata.point_observations.utilities as utilities

#### Data sources that can be accessed with this function currently include:

<table>
    <thead>
        <tr>
            <th>data_source</th>
            <th>variable</th>
            <th>temporal_resolution</th>
            <th>aggregation</th>
            <th>depth_level</th>
            <th>units</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td rowspan=5>'usgs_nwis'</td>
            <td rowspan=2>'streamflow'</td>
            <td>'hourly'</td>
            <td>'average'</td>
            <td></td>
            <td>m^3/s</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'average'</td>
            <td></td>
            <td>m^3/s</td>
        </tr>
        <tr>
            <td rowspan=3>'wtd'</td>
            <td>'hourly'</td>
            <td>'average'</td>
            <td></td>
            <td>m</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'average'</td>
            <td></td>
            <td>m</td>
        </tr>
        <tr>
            <td>'instantaneous'</td>
            <td>'instantaneous'</td>
            <td></td>
            <td>m</td>
        </tr>
        <tr>
            <td rowspan=8>'usda_nrcs'</td>
            <td rowspan=1>'swe'</td>
            <td>'daily'</td>
            <td>'start-of-day'</td>
            <td></td>
            <td>mm</td>
        </tr>
        <tr>
            <td rowspan=3>'precipitation'</td>
            <td>'daily'</td>
            <td>'accumulated'</td>
            <td></td>
            <td>mm</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'total'</td>
            <td></td>
            <td>mm</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'total, snow-adjusted'</td>
            <td></td>
            <td>mm</td>
        </tr>
        <tr>
            <td rowspan=3>'temperature'</td>
            <td>'daily'</td>
            <td>'minimum'</td>
            <td></td>
            <td>C</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'maximum'</td>
            <td></td>
            <td>C</td>
        </tr>
        <tr>
            <td>'daily'</td>
            <td>'average'</td>
            <td></td>
            <td>C</td>
        </tr>
        <tr>
            <td rowspan=1>'soil moisture'</td>
            <td>'daily'</td>
            <td>'start-of-day'</td>
            <td>2, 4, 8, 20, or 40 (inches)</td>
            <td>pct</td>
        </tr>
        <tr>
            <td rowspan=8>'ameriflux'</td>
            <tr>
                <td rowspan=1>'latent heat flux'</td>
                <td>'hourly'</td>
                <td>'total'</td>
                <td></td>
                <td>W/m^2</td>
            </tr>
            <tr>
                <td rowspan=1>'sensible heat flux'</td>
                <td>'hourly'</td>
                <td>'total'</td>
                <td></td>
                <td>W/m^2</td>
            </tr>
            <tr>
                <td rowspan=1>'shortwave radiation'</td>
                <td>'hourly'</td>
                <td>'average'</td>
                <td></td>
                <td>W/m^2</td>
            </tr>
            <tr>
                <td rowspan=1>'longwave radiation'</td>
                <td>'hourly'</td>
                <td>'average'</td>
                <td></td>
                <td>W/m^2</td>
            </tr>
            <tr>
                <td rowspan=1>'vapor pressure deficit'</td>
                <td>'hourly'</td>
                <td>'average'</td>
                <td></td>
                <td>hPa</td>
            </tr>
            <tr>
                <td rowspan=1>'temperature'</td>
                <td>'hourly'</td>
                <td>'average'</td>
                <td></td>
                <td>C</td>
            </tr>
            <tr>
                <td rowspan=1>'wind speed'</td>
                <td>'hourly'</td>
                <td>'average'</td>
                <td></td>
                <td>m/s</td>
            </tr>
        </tr>
    </tbody>
</table>

Note that the field, *depth_level*, needs only be provided when querying soil moisture data.


### Mandatory Parameters (from the above table)
 - data_source
 - variable
 - temporal_resolution
 - aggregation
 - depth_level (only if data_source='usda_nrcs' and variable='soil moisture')
 
### Optional Parameters
 - date_start (as a string of format: 'YYYY-MM-DD')
 - date_end (as a string of format: 'YYYY-MM-DD')
 - state (2-digit state abbreviation, eg: 'NJ')
 - site_ids (list of specific site ID value(s), if known, eg: ['01010000'])
 - latitude_range
 - longitude_range
 
 - return_metadata (default=False; set to True if additional DataFrame containing site metadata information to be returned. Note that the default is to only return a subset of metadata that is available, things like site name, latitude, longitude)
 - all_attributes (default=False; set to True if full set of site metadata we have available should be returned)
 

## Example 1: Specifying a date range and geographic bounding box
#### In this example, a specific start and end date are provided along with a geographic domain. Start and end dates, if provided, must be in 'YYYY-MM-DD' format. If `date_start` is not provided, data is returned from as early as it is available. Likewise, if `date_end` is not provided, data is returned through as current as is available. 

#### Geographic domain specfications, if provided, can be in the form of `latitude_range`, `longitude_range`, `state` (a 2-digit state postal code), or a specific list of site IDs in `site_ids` (example 4 below shows this). If no geography restriction is included, sites from the entire contintental United States will be returned. 

In [ ]:
# Let's first set variables to contain our desired parameters. 
# We will use these for several of the examples
date_start = '2002-01-01'
date_end = '2002-01-05'
latitude_range = (45, 50)
longitude_range = (-75, -65)

In [ ]:
# Store the data observations in 'df'
df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                       temporal_resolution='daily', aggregation='average',
                                       date_start=date_start, date_end=date_end,
                                       latitude_range=latitude_range, longitude_range=longitude_range)
# Inspect results
df.head(10)

### Example 1.5: Returning WTD data for the same geographic region and date range
#### Let's say that now you want groundwater observations for the same geographic region and date range as in the above query. All you have to do is re-define the `data_source`, `variable`, `temporal_resolution`, and `aggregation` for your new data type

In [ ]:
# Store the data observations in 'df'
df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='wtd', 
                                       temporal_resolution='daily', aggregation='average',
                                       date_start=date_start, date_end=date_end,
                                       latitude_range=latitude_range, longitude_range=longitude_range)
# Inspect results
df.head(10)

### Example 2: Returning metadata (select attributes)
#### The default function will return a single DataFrame with the data requested. However, if you set `return_metadata=True`, then a second DataFrame will also be returned. This second DataFrame includes the metadata information relevant to the sites for which data was returned, accomodating for all filtering. As-is, only a select subset of metadata attributes will be included. See Example 3 below for how to return the full set of metadata attributes.

#### Note that the `record_count` in the metadata identifies the full number of records that are avaiable to be queried. The field `num_obs` in the data identifies the number of non-NaN records in your specific query (factoring in any potential date ranges supplied).

In [ ]:
# Store the data observations in 'df' and the metadata information in 'metadata_df'
df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                                    temporal_resolution='daily', aggregation='average',
                                                    date_start=date_start, date_end=date_end,
                                                    latitude_range=latitude_range, longitude_range=longitude_range,
                                                    return_metadata=True)

In [ ]:
# Inspect data returned
df.head(10)

In [ ]:
# Inspect metadata returned
metadata_df.head(10)

### Example 3: Returning metadata (full attributes)
#### The only difference from the above is to indicate *all_attributes=True* to get the full set of metadata attributes we have available for the selected sites.

In [ ]:
# Store the data observations in 'df' and the metadata information in 'metadata_df'
df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                                    temporal_resolution='daily', aggregation='average', 
                                                    date_start=date_start, date_end=date_end,
                                                    latitude_range=latitude_range, longitude_range=longitude_range,
                                                    return_metadata=True, all_attributes=True)

In [ ]:
# Inspect data returned
df.head(10)

In [ ]:
# Inspect metadata returned
metadata_df.head(10)

In [ ]:
# See a list of all metadata attributes that are available
metadata_df.columns

### Example 4: Specifying a specific site ID (no time restriction), or list of site IDs
#### Instead of latitude/longitude bounds, data for a specific stream gage or groundwater well can be returned with or without a date bound. Below, daily streamflow data is returned for a single site and then a select list of sites. There is no time restriction in these examples, so all data available in-house is included.

In [ ]:
point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                  temporal_resolution='daily', aggregation='average', 
                                  site_ids=['01013500'])

In [ ]:
point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                  temporal_resolution='daily', aggregation='average', 
                                  site_ids=['01013500', '01011000', '01029500'])

### Example 7: Obtain citation information for the data source
#### Return general guidance on if/how to cite data for each data source.
#### Where site-specific citations are necessary (currently only AmeriFlux sites require this), an additional parameter site list can be provided to return site-specific data DOI information.

In [ ]:
point_obs.get_citation_information(data_source='usgs_nwis')

In [ ]:
point_obs.get_citation_information(data_source='ameriflux', site_ids=['US-Ne1'])